# Review export: compact tracks for the Pass Review web app

Writes a small tracks file per 5-minute segment (about 2 MB each) so Claude can bring it into the
Pass Review web app. The video is not copied: in the app you open each segment's clip from Drive on the
tablet. Output: `MyDrive/Playbook/review/match_video_2/<segment>/tracks_small.csv.gz` and `segment.json`.

**Runtime: CPU is enough** (Runtime -> Change runtime type -> CPU), about 3 minutes for the whole match.
**Runtime -> Run all**; do not edit cells. Finished segments are skipped.

In [ ]:
# Cell 1 — Drive and inputs
import os, json, glob, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
MATCH = 'match_video_2'
FULL = f'{PBD}/full_match/{MATCH}'
CLIPS = f'{PBD}/video_downloads/{MATCH}_clips'
OUT = f'{PBD}/review/{MATCH}'
if not os.path.exists(f'{FULL}/match_tracks.csv.gz'):
    raise SystemExit(f'Missing in Drive: {FULL}/match_tracks.csv.gz (run notebooks/full_match.ipynb first)')
SEGMENTS = sorted(os.path.basename(c)[:-4] for c in glob.glob(f'{CLIPS}/*.mp4'))
print(len(SEGMENTS), 'segments')

In [ ]:
# Cell 2 — Compact tracks per segment (match frames, so names and joined players match the full match)
import pandas as pd, numpy as np, cv2
COLS = ['frame', 'segment', 'segment_frame', 'raw_tracker_id', 'joined_id', 'class_id', 'team_id',
        'x1', 'y1', 'x2', 'y2', 'x_m', 'y_m']
todo = [s for s in SEGMENTS if not os.path.exists(f'{OUT}/{s}/segment.json')]
if todo:
    t = pd.read_csv(f'{FULL}/match_tracks.csv.gz', low_memory=False, usecols=lambda c: c in COLS)
    for s in todo:
        g = t[t.segment == s].drop(columns='segment').copy()
        for c in ('x1', 'y1', 'x2', 'y2'):
            g[c] = g[c].round().astype(int)
        for c in ('x_m', 'y_m'):
            g[c] = g[c].round()
        os.makedirs(f'{OUT}/{s}', exist_ok=True)
        g.to_csv(f'{OUT}/{s}/tracks_small.csv.gz', index=False, compression='gzip')
        cap = cv2.VideoCapture(f'{CLIPS}/{s}.mp4')
        info = {'segment': s, 'seg_start': int(g.frame.min() - g.segment_frame.min()),
                'frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 'fps': cap.get(cv2.CAP_PROP_FPS),
                'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), 'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}
        cap.release()
        json.dump(info, open(f'{OUT}/{s}/segment.json', 'w'))
        print(s, f"{os.path.getsize(f'{OUT}/{s}/tracks_small.csv.gz') / 1e6:.1f} MB", info)
    del t
print(f'\nDone. Tell Claude the review export is in {OUT}.')